# N0 — Ingesta y preprocesamiento del corpus (Procesamiento)
## AI-MELT: Nivel 0 — Discurso (Corpus)

**Propósito:** Convertir el corpus crudo (Informe Final de la Comisión de la Verdad de Colombia) en un formato estructurado y preprocesado, listo para alimentar el pipeline AI-MELT.

**Entrada:** Archivos PDF o TXT del Informe Final  
**Salida:** `n0_corpus.csv` — una fila por oración con metadatos y anotaciones lingüísticas

**Visualización:** Ejecutar `N0_corpus_ingestion_viz.ipynb` después de este notebook.

---

### Estructura
1. Configuración e instalación de dependencias
2. Carga de datos: definición de rutas y metadatos del corpus
3. Extracción de texto y detección automática de capítulos
3B. Limpieza del texto (encabezados, pies de página, portadas, TOC)
3C. Inspección de la limpieza
4. Segmentación en oraciones
5. Preprocesamiento lingüístico (tokenización, POS, NER, lematización)
6. Generación del DataFrame N0
7. Exportación de resultados

## 1. Configuración e instalación de dependencias

Ejecuta esta celda una vez al inicio de la sesión de Colab.

In [ ]:
# ============================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# Descomenta las líneas siguientes si aún no tienes las dependencias instaladas:
# !pip install PyMuPDF spacy wordcloud matplotlib seaborn pandas tqdm
# !python -m spacy download es_core_news_lg

print("✓ Verificando dependencias...")
import importlib

for pkg in ["fitz", "spacy", "pandas", "wordcloud", "matplotlib", "seaborn", "tqdm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f"  ⚠ Falta: {pkg} — ejecuta pip install para instalarlo")
        raise
print("✓ Todas las dependencias disponibles")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import gc
import json
import os
import re
import warnings
from collections import Counter
from pathlib import Path

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# Cargar modelo de spaCy
# Cargar spaCy con solo los componentes necesarios para reducir uso de memoria
nlp = spacy.load("es_core_news_lg", exclude=["textcat", "custom"])
# Aumentar límite de caracteres por documento (algunos capítulos son largos)
nlp.max_length = 3_000_000

print(f"✓ spaCy modelo cargado: {nlp.meta['name']} (v{nlp.meta['version']})")
print(f"  Pipeline: {nlp.pipe_names}")

## 2. Carga de datos: rutas y metadatos del corpus

### Instrucciones
1. Coloca tus archivos PDF del Informe Final en un directorio local
2. Ajusta la variable `CORPUS_DIR` a la ruta de tu directorio
4. Define los metadatos de cada documento en la variable `METADATA`

> **Nota:** Si tus archivos son TXT en lugar de PDF, el notebook los detectará automáticamente.

In [ ]:
# ============================================================
# 2B. CONFIGURACIÓN DE RUTAS
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  MODIFICA ESTA SECCIÓN CON TUS DATOS                    │
# └─────────────────────────────────────────────────────────┘

# Directorio donde están los PDFs/TXTs del Informe
CORPUS_DIR = "./corpus/"

# Directorio de salida para los resultados
OUTPUT_DIR = "./outputs/N0/"

# Crear directorio de salida si no existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Metadatos del corpus
CORPUS_METADATA = {
    "ID_corpus": "CEV-IF-2022",
    "nombre_corpus": "Informe Final de la Comisión de la Verdad de Colombia",
    "comunidad_discursiva": "CEV, víctimas, actores armados y sociedad colombiana en general",
    "genero_textual": "Informe institucional de comisión de la verdad",
    "fecha": "2022",
    "idioma": "español"
}

# Metadatos por volumen (cada PDF es un volumen con múltiples capítulos adentro).
# Los capítulos se detectan automáticamente vía TOC/bookmarks o tamaño de fuente.
# Aquí solo describes el volumen, no los capítulos.
# Formato: nombre_archivo.pdf -> {titulo, autor, fecha, tipo_documento}
DOCUMENT_METADATA = {
    "1.IF_CONVOCATORIA-A-LA-PAZ-GRANDE_DIGITAL_2022.pdf": {
        "titulo": "Convocatoria a la paz grande",
        "autor": "Comisión de la Verdad",
        "fecha": "2022-06-28",
        "tipo_documento": "Volumen del Informe Final"
    },
    "6 CEV_MI CUERPO ES LA VERDAD_DIGITAL_2022.pdf": {
        "titulo": "Mi cuerpo es la verdad",
        "autor": "Comisión de la Verdad",
        "fecha": "2022-06-28",
        "tipo_documento": "Volumen del Informe Final"
    },
    # Añade más volúmenes aquí...
}

print("✓ Configuración cargada")
print(f"  Corpus: {CORPUS_METADATA['nombre_corpus']}")
print(f"  Directorio: {os.path.abspath(CORPUS_DIR)}")
print(f"  Salida: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
# ============================================================
# 2C. DESCUBRIMIENTO AUTOMÁTICO DE ARCHIVOS
# ============================================================

corpus_path = Path(CORPUS_DIR)
pdf_files = sorted(corpus_path.glob("*.pdf"))
txt_files = sorted(corpus_path.glob("*.txt"))
all_files = pdf_files + txt_files

print("Archivos encontrados:")
print(f"  PDFs: {len(pdf_files)}")
print(f"  TXTs: {len(txt_files)}")
print(f"  Total: {len(all_files)}")
print()

# Listar archivos
for i, f in enumerate(all_files, 1):
    has_meta = "✓ metadata" if f.name in DOCUMENT_METADATA else "⚠ sin metadata (se generará automáticamente)"
    print(f"  {i}. {f.name} — {has_meta}")

if len(all_files) == 0:
    print("\n⚠ No se encontraron archivos. Verifica la ruta CORPUS_DIR.")

## 3. Extracción de texto y detección automática de capítulos

Se detectan los capítulos del documento mediante una cascada de tres estrategias:

1. **TOC/Bookmarks del PDF** (más fiable): `doc.get_toc()` lee la tabla de contenidos interna del PDF. Se toman solo las entradas de nivel 1 (el nivel más exterior/grueso).

2. **Detección tipográfica** (fallback): Se analiza el tamaño de fuente de cada línea. Las líneas con fuente significativamente más grande que el cuerpo del texto se tratan como títulos de capítulo.

3. **Nombre del archivo** (último recurso): Si no hay TOC ni fuentes detectables, se usa el nombre del archivo como nombre del capítulo.

El campo `capitulo` contendrá siempre el **nombre textual** del capítulo (no un número).

In [ ]:
# ============================================================
# 3. EXTRACCIÓN DE TEXTO + DETECCIÓN DE CAPÍTULOS
# ============================================================

def get_chapter_map_from_toc(pdf_path):
    """
    Estrategia 1: Usa el outline/bookmarks internos del PDF.
    Devuelve un dict {página: nombre_capítulo} solo para el nivel más grueso (nivel 1).
    """
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()  # Lista de [nivel, título, página]
    doc.close()

    if not toc:
        return None

    # Filtrar solo nivel 1 (el más exterior)
    level1_entries = [(titulo.strip(), pagina) for nivel, titulo, pagina in toc if nivel == 1]

    if not level1_entries:
        # Si no hay nivel 1, tomar el nivel mínimo disponible
        min_level = min(nivel for nivel, _, _ in toc)
        level1_entries = [(titulo.strip(), pagina) for nivel, titulo, pagina in toc if nivel == min_level]

    if not level1_entries:
        return None

    # Construir mapa: para cada página, asignar el capítulo vigente
    chapter_map = {}
    for i, (titulo, pagina_inicio) in enumerate(level1_entries):
        pagina_fin = level1_entries[i + 1][1] - 1 if i + 1 < len(level1_entries) else 999999
        for p in range(pagina_inicio, pagina_fin + 1):
            chapter_map[p] = titulo

    return chapter_map


def get_chapter_map_from_fonts(pdf_path, top_percentile=0.05):
    """
    Estrategia 2: Detecta títulos de capítulo por tamaño de fuente.
    Las líneas con fuente significativamente más grande que el cuerpo
    y que aparecen al inicio de una sección se tratan como títulos de capítulo.
    """
    doc = fitz.open(pdf_path)
    all_font_sizes = []
    page_texts_with_fonts = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]
        page_lines = []
        for block in blocks:
            if "lines" not in block:
                continue
            for line in block["lines"]:
                spans = line.get("spans", [])
                if not spans:
                    continue
                # Tamaño de fuente dominante de la línea
                max_size = max(s.get("size", 0) for s in spans)
                text = " ".join(s.get("text", "") for s in spans).strip()
                is_bold = any("bold" in s.get("font", "").lower() for s in spans)
                if text and len(text) > 2:
                    all_font_sizes.append(max_size)
                    page_lines.append({
                        "text": text,
                        "size": max_size,
                        "bold": is_bold,
                        "page": page_num + 1
                    })
        page_texts_with_fonts.append(page_lines)

    doc.close()

    if not all_font_sizes:
        return None

    # Determinar umbral: las fuentes en el top percentile son títulos
    sizes_array = np.array(all_font_sizes)
    threshold = np.percentile(sizes_array, (1 - top_percentile) * 100)

    # Extraer candidatos a título de capítulo
    chapter_candidates = []
    for page_lines in page_texts_with_fonts:
        for line in page_lines:
            if line["size"] >= threshold and len(line["text"]) > 3 and len(line["text"]) < 200:
                # Filtrar líneas que parecen ser números de página o headers repetidos
                text = line["text"].strip()
                if re.match(r'^\d{1,4}$', text):
                    continue
                chapter_candidates.append({
                    "text": text,
                    "page": line["page"],
                    "size": line["size"],
                    "bold": line["bold"]
                })

    if not chapter_candidates:
        return None

    # Agrupar candidatos consecutivos (a veces un título ocupa varias líneas)
    # y tomar el texto más largo como nombre del capítulo
    chapters = []
    current_chapter = chapter_candidates[0]
    for cand in chapter_candidates[1:]:
        if cand["page"] == current_chapter["page"] and cand["size"] == current_chapter["size"]:
            # Mismo tamaño en la misma página: concatenar (título multi-línea)
            current_chapter["text"] += " " + cand["text"]
        else:
            chapters.append(current_chapter)
            current_chapter = cand
    chapters.append(current_chapter)

    # Construir mapa
    chapter_map = {}
    for i, ch in enumerate(chapters):
        page_end = chapters[i + 1]["page"] - 1 if i + 1 < len(chapters) else 999999
        # Limpiar el nombre: quitar numeración inicial si existe, capitalizar
        name = ch["text"].strip()
        # Quitar patrones como "Capítulo 1. ", "1. ", "I. " al inicio
        name = re.sub(r'^(Capítulo|Cap\.?)\s*\d+\.?\s*', '', name, flags=re.IGNORECASE)
        name = re.sub(r'^\d+\.\s*', '', name)
        name = re.sub(r'^[IVXLC]+\.\s*', '', name)
        name = name.strip()
        if name:
            for p in range(ch["page"], page_end + 1):
                chapter_map[p] = name

    return chapter_map


def get_chapter_for_page(page_num, chapter_map, fallback_name):
    """Obtiene el nombre del capítulo para una página dada."""
    if chapter_map and page_num in chapter_map:
        return chapter_map[page_num]
    return fallback_name


def extract_text_from_pdf(pdf_path):
    """Extrae texto de un PDF página por página."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        if text.strip():
            pages.append({
                "pagina": page_num + 1,
                "texto": text.strip()
            })
    doc.close()
    return pages


def extract_text_from_txt(txt_path):
    """Lee un archivo TXT y lo trata como una sola 'página'."""
    with open(txt_path, encoding='utf-8') as f:
        text = f.read()
    return [{"pagina": 1, "texto": text.strip()}]


# ── Extraer texto y detectar capítulos ──
all_pages = []

for filepath in tqdm(all_files, desc="Extrayendo texto + capítulos"):
    # Metadatos del documento
    meta = DOCUMENT_METADATA.get(filepath.name, {
        "titulo": filepath.stem.replace("_", " ").title(),
        "autor": "Comisión de la Verdad",
        "fecha": "2022",
        "tipo_documento": "Volumen del Informe Final"
    })
    ID_documento = f"DOC-{filepath.stem}"

    # --- Detección de capítulos (cascada) ---
    chapter_map = None
    chapter_method = "ninguno"

    if filepath.suffix.lower() == '.pdf':
        # Estrategia 1: TOC/bookmarks
        chapter_map = get_chapter_map_from_toc(filepath)
        if chapter_map:
            chapter_method = "toc_bookmarks"
            unique_chapters = sorted(set(chapter_map.values()), key=lambda x: min(p for p, c in chapter_map.items() if c == x))
            print(f"  {filepath.name}: {len(unique_chapters)} capítulos detectados vía TOC")
            for ch in unique_chapters:
                pages_range = sorted([p for p, c in chapter_map.items() if c == ch])
                print(f"    • pp. {pages_range[0]}-{pages_range[-1]}: {ch}")
        else:
            # Estrategia 2: Detección tipográfica
            chapter_map = get_chapter_map_from_fonts(filepath, top_percentile=0.03)
            if chapter_map:
                chapter_method = "font_size"
                unique_chapters = sorted(set(chapter_map.values()), key=lambda x: min(p for p, c in chapter_map.items() if c == x))
                print(f"  {filepath.name}: {len(unique_chapters)} capítulos detectados vía tamaño de fuente")
                for ch in unique_chapters:
                    pages_range = sorted([p for p, c in chapter_map.items() if c == ch])
                    print(f"    • pp. {pages_range[0]}-{pages_range[-1]}: {ch}")
            else:
                chapter_method = "fallback_filename"
                print(f"  {filepath.name}: sin TOC ni fuentes detectables → fallback al nombre del archivo")

    # Fallback: nombre del archivo o metadato manual
    # Fallback: si no se detectan capítulos, usar el título del volumen como capítulo
    fallback_chapter = meta.get("titulo", "") or filepath.stem.replace("_", " ").title()

    # --- Extraer texto ---
    if filepath.suffix.lower() == '.pdf':
        pages = extract_text_from_pdf(filepath)
    else:
        pages = extract_text_from_txt(filepath)

    for page in pages:
        capitulo_detectado = get_chapter_for_page(page["pagina"], chapter_map, fallback_chapter)
        all_pages.append({
            "ID_documento": ID_documento,
            "archivo": filepath.name,
            "volumen": meta.get("titulo", filepath.stem.replace("_", " ").title()),
            "capitulo": capitulo_detectado,
            "pagina": page["pagina"],
            "texto_pagina": page["texto"],
            "n_caracteres": len(page["texto"]),
            "metodo_capitulo": chapter_method
        })

df_pages = pd.DataFrame(all_pages)
print(f"\n✓ Texto extraído: {len(df_pages)} páginas de {len(all_files)} documentos")
print(f"  Total caracteres: {df_pages['n_caracteres'].sum():,}")
print("\n  Métodos de detección de capítulos usados:")
for method, count in df_pages["metodo_capitulo"].value_counts().items():
    print(f"    {method}: {count} páginas")
print("\n  Capítulos detectados:")
for cap in df_pages["capitulo"].unique():
    n_pages = len(df_pages[df_pages["capitulo"] == cap])
    print(f"    • {cap} ({n_pages} páginas)")

## 3B. Limpieza del texto extraído

El texto crudo de los PDFs contiene elementos que **no son contenido discursivo** y que contaminarían el análisis metafórico:
- **Encabezados repetidos** en cada página ("CONVOCATORIA A LA PAZ GRANDE", "ESCLARECER LA VERDAD", etc.)
- **Pies de página** con títulos de sección
- **Números de página**
- **Tablas de contenido** (listas de capítulos con números de página)
- **Contraportadas, portadas y créditos**
- **Notas al pie** con numeración
- **Líneas muy cortas** que suelen ser encabezados o fragmentos de diseño

Esta celda detecta y elimina estos elementos de forma automática y configurable.

In [ ]:
# ============================================================
# 3B. LIMPIEZA DE TEXTO — ENCABEZADOS, PIES, PORTADAS, TOC
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  CONFIGURA AQUÍ LOS PATRONES DE TU CORPUS               │
# └─────────────────────────────────────────────────────────┘

# Encabezados y pies de página que se repiten en cada página del PDF.
HEADERS_FOOTERS = [
    "CONVOCATORIA A LA PAZ GRANDE",
    "ESCLARECER LA VERDAD",
    "HAY FUTURO SI HAY VERDAD",
    "HALLAZGOS Y RECOMENDACIONES",
    "NO MATARÁS",
    "HASTA LA GUERRA TIENE LÍMITES",
    "MI CUERPO ES LA VERDAD",
    "CUANDO LOS PÁJAROS NO CANTABAN",
    "RESISTIR NO ES AGUANTAR",
    "SUFRIR LA GUERRA Y REHACER LA VIDA",
    "COLOMBIA ADENTRO",
    "COMISIÓN DE LA VERDAD",
    "INTRODUCCIÓN",
    # Añade más patrones de tu edición...
]

# Páginas a excluir completamente (portadas, contraportadas, créditos, TOC)
# Soporta páginas individuales y rangos con tuplas (inicio, fin) inclusive.
# Formato: {nombre_archivo: [página, página, (rango_inicio, rango_fin), ...]}
PAGES_TO_EXCLUDE = {
    "1.IF_CONVOCATORIA-A-LA-PAZ-GRANDE_DIGITAL_2022.pdf": [(1, 10), (52, 56)],
    "6 CEV_MI CUERPO ES LA VERDAD_DIGITAL_2022.pdf": [(1, 20), (334, 644)],
    # Añade más volúmenes aquí...
}

# Longitud mínima de línea para conservar (en caracteres).
MIN_LINE_LENGTH = 30

# Patrones regex adicionales a eliminar
REGEX_PATTERNS_TO_REMOVE = [
    r'^\d{1,4}$',                      # Números de página solos
    r'^\d{1,4}\s*$',                   # Números de página con espacios
    r'^Página\s+\d+',                  # "Página 28"
    r'^\d+\s+de\s+\d+',               # "28 de 350"
    r'^Capítulo\s+\d+',                # "Capítulo 3" (encabezados de sección)
    r'^Tabla de contenido',             # Tabla de contenido
    r'^Índice',                          # Índice
    r'^Contenido',                       # Contenido
    r'^REFERENCIAS\s*$',                # Referencias
    r'^BIBLIOGRAFÍA\s*$',
    r'^Fuente:',                         # Pies de tabla/gráficos
    r'^Foto:',                           # Créditos de fotos
    r'^Ilustración:',
    r'^Gráfic[oa]\s+\d+',              # "Gráfico 1"
    r'^Tabla\s+\d+',                    # "Tabla 1"
    r'^Mapa\s+\d+',                     # "Mapa 1"
    r'^Figura\s+\d+',                   # "Figura 1"
]


def expand_page_ranges(page_specs):
    """Expande una lista mixta de páginas y rangos en un set de páginas individuales.
    Ejemplo: [(1, 12), 350, (280, 285)] → {1,2,3,...,12,280,281,...,285,350}
    """
    pages = set()
    for spec in page_specs:
        if isinstance(spec, tuple) and len(spec) == 2:
            pages.update(range(spec[0], spec[1] + 1))
        else:
            pages.add(spec)
    return pages


def detect_repeated_headers(pages_text, threshold=0.3):
    """Detecta líneas que se repiten en más del threshold de páginas."""
    n_pages = len(pages_text)
    if n_pages < 3:
        return set()

    line_counts = Counter()
    for page in pages_text:
        unique_lines = set()
        for line in page.split('\n'):
            cleaned = line.strip()
            if 5 < len(cleaned) < 100:
                unique_lines.add(cleaned)
        for line in unique_lines:
            line_counts[line] += 1

    repeated = {line for line, count in line_counts.items()
                if count >= n_pages * threshold}
    return repeated


def clean_page_text(text, headers_set, regex_patterns, min_line_length):
    """Limpia el texto de una página eliminando headers, footers y ruido.
    
    Usa comparación exacta (case-insensitive) contra el set de headers.
    No usa búsqueda de substring para evitar falsos positivos
    (ej: eliminar párrafos que mencionan 'Comisión de la Verdad' en contexto).
    """
    lines = text.split('\n')
    cleaned_lines = []
    headers_upper = {h.upper().strip() for h in headers_set}

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        # Coincidencia exacta (case-insensitive)
        if stripped.upper() in headers_upper:
            continue

        # Patrones regex
        skip = False
        for pattern in regex_patterns:
            if re.match(pattern, stripped, re.IGNORECASE):
                skip = True
                break
        if skip:
            continue

        # Líneas demasiado cortas (excepto si empiezan con minúscula → fin de párrafo)
        if 0 < len(stripped) < min_line_length and not stripped[0].islower():
            continue

        cleaned_lines.append(stripped)

    return '\n'.join(cleaned_lines)

def remove_footnotes_from_bottom(text):
    """Recorre las líneas de abajo hacia arriba. Mientras coincidan con el
    patrón de nota al pie (número + espacio + mayúscula + ... + punto),
    las extrae. Se detiene en la primera línea que no es nota.
    Devuelve (texto_sin_notas, lista_de_notas_extraídas).
    """
    lines = text.split('\n')
    footnotes = []
    while lines:
        last = lines[-1].strip()
        if not last:
            lines.pop()
            continue
        if re.match(r'^\d{1,3}\s+[A-ZÁÉÍÓÚÑ].*\.\s*$', last):
            footnotes.append(last)
            lines.pop()
        else:
            break
    footnotes.reverse()  # Restaurar orden original (de arriba a abajo)
    return '\n'.join(lines), footnotes

# ── Ejecutar limpieza ──

# Paso 1A: Detectar headers POR VOLUMEN (no solo global)
# Esto atrapa headers de sección que solo se repiten dentro de un volumen
# (ej: "conclusiones" que aparece en muchas páginas del volumen 6 pero no en otros).
auto_headers_per_volume = {}
for archivo in df_pages["archivo"].unique():
    pages_of_vol = df_pages[df_pages["archivo"] == archivo]["texto_pagina"].tolist()
    vol_headers = detect_repeated_headers(pages_of_vol, threshold=0.3)
    auto_headers_per_volume[archivo] = vol_headers
    if vol_headers:
        print(f"  Headers auto-detectados en {archivo}: {len(vol_headers)}")
        for h in sorted(vol_headers)[:8]:
            print(f"    • \"{h}\"")
        if len(vol_headers) > 8:
            print(f"    ... y {len(vol_headers) - 8} más")

# Paso 1B: Inyectar nombres de capítulo como headers
# Los títulos de capítulo suelen coincidir con los encabezados de página.
chapter_headers_per_volume = {}
for archivo in df_pages["archivo"].unique():
    vol_pages = df_pages[df_pages["archivo"] == archivo]
    if "capitulo" in vol_pages.columns:
        cap_names = vol_pages["capitulo"].dropna().unique().tolist()
        chapter_headers_per_volume[archivo] = set(cap_names)
        if cap_names:
            print(f"  Capítulos inyectados como headers en {archivo}: {len(cap_names)}")
            for cn in cap_names[:5]:
                print(f"    • \"{cn}\"")
            if len(cap_names) > 5:
                print(f"    ... y {len(cap_names) - 5} más")

# Paso 1C: Headers globales (detección sobre todo el corpus)
all_page_texts = df_pages["texto_pagina"].tolist()
auto_detected_global = detect_repeated_headers(all_page_texts, threshold=0.3)

# Combinar headers globales: manual + auto global
all_headers_global = set(HEADERS_FOOTERS) | auto_detected_global

print("\nResumen de headers:")
print(f"  Manuales: {len(HEADERS_FOOTERS)}")
print(f"  Auto-detectados (global): {len(auto_detected_global)}")
print(f"  Auto-detectados (por volumen): {sum(len(v) for v in auto_headers_per_volume.values())}")
print(f"  Inyectados desde capítulos: {sum(len(v) for v in chapter_headers_per_volume.values())}")


# Paso 2: Excluir páginas completas (portadas, TOC, etc.)
pages_excluded = 0
indices_to_drop = []
expanded_excludes = {archivo: expand_page_ranges(specs) for archivo, specs in PAGES_TO_EXCLUDE.items()}

for idx, row in df_pages.iterrows():
    if row["archivo"] in expanded_excludes and row["pagina"] in expanded_excludes[row["archivo"]]:
        indices_to_drop.append(idx)
        pages_excluded += 1

df_pages_clean = df_pages.drop(index=indices_to_drop).copy()
print(f"\nPáginas excluidas (portadas/TOC): {pages_excluded}")


# Paso 3: Limpiar texto de cada página y extraer notas al pie
# Para cada página, combinar headers globales + headers de su volumen + capítulos de su volumen
compiled_patterns = REGEX_PATTERNS_TO_REMOVE
original_chars = df_pages_clean["texto_pagina"].str.len().sum()

def get_headers_for_page(archivo):
    """Combina todos los headers aplicables a las páginas de un volumen."""
    combined = set(all_headers_global)
    combined |= auto_headers_per_volume.get(archivo, set())
    combined |= chapter_headers_per_volume.get(archivo, set())
    return combined

# Primero: eliminar notas al pie (de abajo hacia arriba)
all_footnotes = []

def process_page(row):
    text = row["texto_pagina"]
    text_clean, footnotes = remove_footnotes_from_bottom(text)
    for fn in footnotes:
        all_footnotes.append({
            "archivo": row["archivo"],
            "pagina": row["pagina"],
            "capitulo": row.get("capitulo", ""),
            "nota_al_pie": fn
        })
    return text_clean

df_pages_clean["texto_pagina"] = df_pages_clean.apply(process_page, axis=1)

# Segundo: limpiar headers/footers/regex
df_pages_clean["texto_limpio"] = df_pages_clean.apply(
    lambda row: clean_page_text(
        row["texto_pagina"],
        get_headers_for_page(row["archivo"]),
        compiled_patterns,
        MIN_LINE_LENGTH
    ),
    axis=1
)

# Eliminar páginas vacías
df_pages_clean = df_pages_clean[df_pages_clean["texto_limpio"].str.len() > 50].copy()
cleaned_chars = df_pages_clean["texto_limpio"].str.len().sum()

print("\nResultado de la limpieza:")
print(f"  Páginas antes:     {len(df_pages)}")
print(f"  Páginas después:   {len(df_pages_clean)}")
print(f"  Caracteres antes:  {original_chars:,}")
print(f"  Caracteres después:{cleaned_chars:,}")
print(f"  Reducción:         {(1 - cleaned_chars/original_chars)*100:.1f}%")
print(f"  Notas al pie extraídas: {len(all_footnotes):,}")

# Exportar notas al pie
df_footnotes = pd.DataFrame(all_footnotes)
if len(df_footnotes) > 0:
    footnotes_path = os.path.join(OUTPUT_DIR, "n0_notas_al_pie.csv")
    df_footnotes.to_csv(footnotes_path, index=False, encoding='utf-8-sig')
    print(f"  ✓ {footnotes_path}")

# Reemplazar df_pages
df_pages = df_pages_clean.copy()
df_pages["texto_pagina"] = df_pages["texto_limpio"]
df_pages = df_pages.drop(columns=["texto_limpio"])
df_pages["n_caracteres"] = df_pages["texto_pagina"].str.len()

print("\n✓ Texto limpio listo para segmentación")




### 3C. Inspección de la limpieza

Revisa muestras del texto limpio para verificar que la limpieza es correcta y que no se eliminó contenido discursivo relevante.

In [ ]:
# ============================================================
# 3C. INSPECCIÓN VISUAL DE LA LIMPIEZA
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  CONFIGURA LA INSPECCIÓN                                 │
# └─────────────────────────────────────────────────────────┘

# --- Modo de selección ---
# "aleatorio"  → muestra aleatoria de INSPECTION_SAMPLE páginas
# "manual"     → páginas específicas definidas en INSPECTION_PAGES
INSPECTION_MODE = "manual"

# --- Configuración para modo aleatorio ---
INSPECTION_SAMPLE = 5

# --- Configuración para modo manual ---
# Formato: {nombre_archivo: [lista de páginas]}
# Soporta rangos con tuplas: [(inicio, fin)]
INSPECTION_PAGES = {
    "6 CEV_MI CUERPO ES LA VERDAD_DIGITAL_2022.pdf": [114],
    "1.IF_CONVOCATORIA-A-LA-PAZ-GRANDE_DIGITAL_2022.pdf": [15, 30],
}

# --- Configuración común ---
INSPECTION_CHARS = 100

# Qué parte del texto mostrar: "inicio", "final" o "ambos"
INSPECTION_POSITION = "ambos"

# Filtrar por documento (solo aplica en modo aleatorio; None = todos)
INSPECTION_DOCS = None
# INSPECTION_DOCS = ["6 CEV_MI CUERPO ES LA VERDAD_DIGITAL_2022.pdf"]

# ── Ejecutar inspección ──

if INSPECTION_MODE == "manual" and INSPECTION_PAGES:
    # Expandir rangos
    sample_indices = []
    for archivo, pages in INSPECTION_PAGES.items():
        expanded = set()
        for spec in pages:
            if isinstance(spec, tuple) and len(spec) == 2:
                expanded.update(range(spec[0], spec[1] + 1))
            else:
                expanded.add(spec)
        matches = df_pages[(df_pages["archivo"] == archivo) & (df_pages["pagina"].isin(expanded))]
        sample_indices.append(matches)
        missing = expanded - set(matches["pagina"].tolist())
        if missing:
            print(f"⚠ Páginas no encontradas en {archivo}: {sorted(missing)}")
    sample_pages = pd.concat(sample_indices) if sample_indices else pd.DataFrame()

elif INSPECTION_MODE == "aleatorio":
    df_inspect = df_pages.copy()
    if INSPECTION_DOCS:
        df_inspect = df_inspect[df_inspect["archivo"].isin(INSPECTION_DOCS)]
        print(f"Filtrando por {len(INSPECTION_DOCS)} documento(s)")
    sample_pages = df_inspect.sample(n=min(INSPECTION_SAMPLE, len(df_inspect)), random_state=42)

else:
    sample_pages = pd.DataFrame()

if len(sample_pages) == 0:
    print("⚠ No hay páginas para inspeccionar con la configuración actual.")
else:
    for _, row in sample_pages.iterrows():
        print("=" * 70)
        print(f"Documento: {row['archivo']} | Página: {row['pagina']}")
        if 'capitulo' in row and row['capitulo']:
            print(f"Capítulo: {row['capitulo']}")
        total_chars = len(row["texto_pagina"])
        print(f"Total: {total_chars:,} caracteres")
        print("=" * 70)

        if INSPECTION_POSITION == "inicio":
            print(row["texto_pagina"][:INSPECTION_CHARS])
            if total_chars > INSPECTION_CHARS:
                print(f"\n... ({total_chars - INSPECTION_CHARS:,} caracteres más)")

        elif INSPECTION_POSITION == "final":
            if total_chars > INSPECTION_CHARS:
                print(f"({total_chars - INSPECTION_CHARS:,} caracteres antes) ...\n")
            print(row["texto_pagina"][-INSPECTION_CHARS:])

        elif INSPECTION_POSITION == "ambos":
            half = INSPECTION_CHARS // 2
            print(row["texto_pagina"][:half])
            if total_chars > INSPECTION_CHARS:
                print(f"\n... ({total_chars - INSPECTION_CHARS:,} caracteres omitidos) ...\n")
            print(row["texto_pagina"][-half:])

        print()

    print(f"✓ Inspección de {len(sample_pages)} páginas (modo: {INSPECTION_MODE}, posición: {INSPECTION_POSITION}).")
    print("  Si ves ruido, añade patrones a HEADERS_FOOTERS o REGEX_PATTERNS_TO_REMOVE y re-ejecuta 3B.")

In [ ]:
# ============================================================
# 3D. INSPECCIÓN DE NOTAS AL PIE EXTRAÍDAS
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  CONFIGURA LA INSPECCIÓN DE NOTAS AL PIE                │
# └─────────────────────────────────────────────────────────┘

# Cuántas notas muestrear
FOOTNOTE_SAMPLE = 20

# Filtrar por documento (None = todos)
FOOTNOTE_DOCS = None
# FOOTNOTE_DOCS = ["6 CEV_MI CUERPO ES LA VERDAD_DIGITAL_2022.pdf"]
# FOOTNOTE_DOCS = ["1.IF_CONVOCATORIA-A-LA-PAZ-GRANDE_DIGITAL_2022.pdf"]


# ── Ejecutar inspección ──

if len(df_footnotes) > 0:
    df_fn_inspect = df_footnotes.copy()
    if FOOTNOTE_DOCS:
        df_fn_inspect = df_fn_inspect[df_fn_inspect["archivo"].isin(FOOTNOTE_DOCS)]

    print(f"Notas al pie extraídas: {len(df_footnotes):,} total")
    print("  Por documento:")
    for archivo, count in df_footnotes["archivo"].value_counts().items():
        print(f"    • {archivo}: {count:,} notas")
    print()

    sample_fn = df_fn_inspect.sample(n=min(FOOTNOTE_SAMPLE, len(df_fn_inspect)), random_state=42)

    print(f"Muestra de {len(sample_fn)} notas al pie:")
    print("-" * 70)
    for _, row in sample_fn.iterrows():
        print(f"  [{row['archivo'][:30]}...] p.{row['pagina']} | {row['nota_al_pie'][:120]}")
    print("-" * 70)
    print()
    print("⚠ Verifica que todas sean notas al pie reales y no texto del cuerpo.")
    print("  Si ves falsos positivos, ajusta el regex en remove_footnotes_from_bottom().")
else:
    print("No se extrajeron notas al pie.")

## 4. Segmentación en oraciones

Se utiliza el sentencizer de spaCy para segmentar el texto en oraciones. Cada oración recibe un identificador único que preserva la trazabilidad (documento → página → oración).

In [ ]:
# ============================================================
# 4. SEGMENTACIÓN EN ORACIONES CON SPACY
# ============================================================

def segment_into_sentences(text, min_length=10, max_length=2000):
    """Segmenta texto en oraciones usando spaCy."""
    doc = nlp(text)
    sentences = []
    for sent in doc.sents:
        sent_text = sent.text.strip()
        # Elimina los saltos de línea en una oración
        sent_text = re.sub(r'\s+', ' ', sent_text) 
        # Filtrar oraciones muy cortas (probablemente encabezados o fragmentos)
        # y muy largas (probablemente errores de segmentación)
        if len(sent_text) >= min_length and len(sent_text) <= max_length:
            sentences.append(sent_text)
    return sentences

# Segmentar todas las páginas
all_sentences = []
sentence_counter = 0

for _, row in tqdm(df_pages.iterrows(), total=len(df_pages), desc="Segmentando oraciones"):
    sentences = segment_into_sentences(row["texto_pagina"])
    for sent_text in sentences:
        sentence_counter += 1
        all_sentences.append({
            "ID_documento": row["ID_documento"],
            "volumen": row["volumen"],
            "capitulo": row["capitulo"],
            "pagina": row["pagina"],
            "ID_oracion": f"S-{sentence_counter:06d}",
            "oracion_texto": sent_text,
            "n_palabras": len(sent_text.split()),
            "n_caracteres": len(sent_text)
        })

df_sentences = pd.DataFrame(all_sentences)
print(f"\n✓ Segmentación completada: {len(df_sentences):,} oraciones")
print(f"  Palabras promedio por oración: {df_sentences['n_palabras'].mean():.1f}")
print(f"  Mediana: {df_sentences['n_palabras'].median():.0f}")

## 5. Preprocesamiento lingüístico

Para cada oración se extraen:
- **Tokens:** lista de palabras
- **Lemas:** forma canónica de cada token
- **POS tags:** categoría gramatical (VERB, NOUN, ADJ, etc.)
- **Entidades NER:** personas, organizaciones, lugares mencionados

> **Nota:** Este paso puede tomar varios minutos dependiendo del tamaño del corpus. Se procesan las oraciones por lotes para optimizar.

In [ ]:
# ============================================================
# 5. PREPROCESAMIENTO LINGÜÍSTICO CON SPACY (optimizado para memoria)
# ============================================================

def process_batch(texts_batch, indices_batch):
    """Procesa un lote de oraciones y devuelve resultados como listas ligeras."""
    batch_results = []
    for doc in nlp.pipe(texts_batch, batch_size=200, n_process=1):
        tokens = [t.text for t in doc if not t.is_space]
        lemas = [t.lemma_ for t in doc if not t.is_space]
        pos_tags = [t.pos_ for t in doc if not t.is_space]
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        batch_results.append((tokens, lemas, pos_tags, entities))
    return batch_results

# Procesar por lotes para controlar memoria
BATCH_SIZE = 500
all_tokens = []
all_lemas = []
all_pos = []
all_ner = []

n_total = len(df_sentences)
for start in tqdm(range(0, n_total, BATCH_SIZE), desc="Procesando NLP"):
    end = min(start + BATCH_SIZE, n_total)
    texts_batch = df_sentences["oracion_texto"].iloc[start:end].tolist()
    indices_batch = list(range(start, end))

    batch_results = process_batch(texts_batch, indices_batch)

    for tokens, lemas, pos_tags, entities in batch_results:
        # Guardar como JSON strings compactos (menor memoria que listas Python)
        all_tokens.append(json.dumps(tokens, ensure_ascii=False, separators=(',', ':')))
        all_lemas.append(json.dumps(lemas, ensure_ascii=False, separators=(',', ':')))
        all_pos.append(json.dumps(pos_tags, ensure_ascii=False, separators=(',', ':')))
        all_ner.append(json.dumps([{"text": t, "label": l} for t, l in entities],
                                   ensure_ascii=False, separators=(',', ':')))

    # Liberar memoria del lote
    del batch_results, texts_batch
    gc.collect()

df_sentences["tokens"] = all_tokens
df_sentences["lemas"] = all_lemas
df_sentences["pos_tags"] = all_pos
df_sentences["entidades_NER"] = all_ner

# Liberar listas temporales
del all_tokens, all_lemas, all_pos, all_ner
gc.collect()

print(f"\n✓ Preprocesamiento completado para {len(df_sentences):,} oraciones")
print(f"  Memoria DataFrame: {df_sentences.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

## 6. DataFrame N0 — Resultado final

El DataFrame N0 contiene una fila por oración con todos los metadatos y anotaciones lingüísticas.

In [ ]:
# ============================================================
# 6. CONSTRUCCIÓN DEL DATAFRAME N0
# ============================================================

# Añadir metadatos del corpus
df_n0 = df_sentences.copy()
df_n0.insert(0, "ID_corpus", CORPUS_METADATA["ID_corpus"])

# Vista previa
print(f"DataFrame N0: {df_n0.shape[0]:,} filas × {df_n0.shape[1]} columnas")
print(f"\nColumnas: {list(df_n0.columns)}")
print(f"\nDocumentos únicos: {df_n0['ID_documento'].nunique()}")
print(f"Volúmenes únicos: {df_n0['volumen'].nunique()}")
print(f"Capítulos únicos: {df_n0['capitulo'].nunique()}")
print()
df_n0.head(3)

## 7. Exportación de resultados

In [ ]:
# ============================================================
# 7. EXPORTACIÓN
# ============================================================

# CSV principal
output_csv = os.path.join(OUTPUT_DIR, "n0_corpus.csv")
df_n0.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✓ Exportado: {output_csv}")
print(f"  Tamaño: {os.path.getsize(output_csv) / 1024 / 1024:.1f} MB")

# Metadatos del corpus en JSON
metadata_path = os.path.join(OUTPUT_DIR, "n0_metadata.json")
import datetime

metadata = {
    **CORPUS_METADATA,
    "timestamp": datetime.datetime.now().isoformat(),
    "stats": {
        "n_documentos": int(df_n0["ID_documento"].nunique()),
        "n_oraciones": len(df_n0),
        "n_palabras_total": int(df_n0["n_palabras"].sum()),
        "palabras_promedio_por_oracion": round(float(df_n0["n_palabras"].mean()), 1)
    },
    "archivos_procesados": [f.name for f in all_files],
    "spacy_model": f"{nlp.meta['lang']}_{nlp.meta['name']}",
    "spacy_version": nlp.meta["version"]
}
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ Metadatos: {metadata_path}")

# Resumen de documentos
doc_summary = df_n0.groupby(["ID_documento", "volumen", "capitulo"]).agg(
    n_oraciones=("ID_oracion", "count"),
    n_palabras=("n_palabras", "sum"),
    paginas=("pagina", "nunique")
).reset_index()
doc_summary_path = os.path.join(OUTPUT_DIR, "n0_resumen_documentos.csv")
doc_summary.to_csv(doc_summary_path, index=False, encoding='utf-8-sig')
print(f"✓ Resumen: {doc_summary_path}")

print(f"\n{'='*60}")
print("RESUMEN DEL CORPUS N0")
print(f"{'='*60}")
print(f"Documentos: {metadata['stats']['n_documentos']}")
print(f"Oraciones:  {metadata['stats']['n_oraciones']:,}")
print(f"Palabras:   {metadata['stats']['n_palabras_total']:,}")
print(f"Promedio:   {metadata['stats']['palabras_promedio_por_oracion']} palabras/oración")

In [ ]:
## 9. Resumen y siguiente paso